# GraphRAG with Neo4j Knowledge Graph

> **Teaching goal:** this notebook is designed as a step-by-step, instructor-friendly Colab lab. Every important cell explains what it does, why it matters, and how it fits into RAG / GraphRAG.

## Corpus used in this lab
We use public research papers as a realistic mini-corpus:

1. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al.
2. **From Local to Global: A Graph RAG Approach to Query-Focused Summarization** — Edge et al.

These papers are useful because they naturally contain entities, methods, datasets, claims, and relationships. That makes them good for comparing:

- **Traditional RAG:** retrieve semantically similar chunks.
- **GraphRAG:** retrieve facts and relationships from a knowledge graph.
- **Hybrid RAG:** combine vector retrieval and graph traversal.

## What this notebook demonstrates

This notebook builds a real **graph database** workflow with Neo4j.

You will learn:

1. How to extract entities and relationships from research papers.
2. How to convert extracted facts into triples.
3. How to load those triples into Neo4j.
4. How to query the graph using Cypher.
5. How to answer relationship/path questions using graph context.

GraphRAG is strongest when the question depends on **relationships**, **paths**, **communities**, or **multi-hop reasoning**.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# In Colab, run this cell first. It installs:
# - openai: LLM + embeddings
# - pymupdf: PDF text extraction
# - pandas/numpy/tqdm: data handling
# - tiktoken: optional token-aware splitting

!pip -q install openai pymupdf pandas numpy tqdm tiktoken

!pip -q install neo4j networkx pyvis

In [ ]:
# ============================================================
# 2. Configure API keys safely
# ============================================================
# Recommended in Colab:
#   Left sidebar -> Secrets -> add OPENAI_API_KEY
# Then this cell will read it without exposing the key in the notebook.

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    # Not running in Colab. You can set your key in your environment instead.
    pass

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in Colab Secrets or as an environment variable.")

from openai import OpenAI
client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("OpenAI client configured. Models:", CHAT_MODEL, EMBED_MODEL)

In [ ]:
# ============================================================
# 3. Common helper functions: LLM + embeddings
# ============================================================
# llm() is used for answer generation, extraction, routing, and evaluation.
# embed_texts() converts a list of strings into embedding vectors.

import numpy as np

def llm(prompt, system="You are a precise, helpful assistant.", temperature=0.0):
    """Single-turn LLM call. Returns plain text."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content.strip()


def embed_texts(texts, batch_size=64):
    """Embed a string or list of strings using OpenAI embeddings."""
    if isinstance(texts, str):
        texts = [texts]
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in response.data])
    return np.array(vectors, dtype="float32")

print(llm("Reply with exactly: ready"))
print("Embedding shape for one text:", embed_texts("hello").shape)

In [ ]:
# ============================================================
# 4. Download public research papers and extract text
# ============================================================
# We download PDFs from public URLs and extract text with PyMuPDF.
# The resulting `documents` list will be used by RAG, GraphRAG, and Hybrid RAG.

import requests, fitz, re
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path("/content/rag_graphrag_corpus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = [
    {
        "doc_id": "rag_lewis_2020",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url": "https://arxiv.org/pdf/2005.11401",
    },
    {
        "doc_id": "graphrag_edge_2024",
        "title": "From Local to Global: A Graph RAG Approach to Query-Focused Summarization",
        "url": "https://arxiv.org/pdf/2404.16130",
    },
]


def download_pdf(url, path):
    if path.exists() and path.stat().st_size > 10_000:
        return path
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def extract_pdf_text(path):
    doc = fitz.open(path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"page": page_num, "text": text})
    return pages

# Build a page-level document list.
documents = []
for paper in tqdm(PAPERS):
    pdf_path = DATA_DIR / f"{paper['doc_id']}.pdf"
    download_pdf(paper["url"], pdf_path)
    for page in extract_pdf_text(pdf_path):
        documents.append({
            "doc_id": paper["doc_id"],
            "title": paper["title"],
            "page": page["page"],
            "text": page["text"],
        })

print(f"Loaded {len(documents)} page-documents")
print(documents[0].keys())
print(documents[0]["title"], "page", documents[0]["page"])
print(documents[0]["text"][:500])

In [ ]:
# ============================================================
# 5. Chunk the corpus
# ============================================================
# Why chunk?
# LLMs and vector databases work better with small passages than entire PDFs.
# Each chunk keeps metadata so we can cite the source paper and page.

import textwrap


def chunk_text(text, chunk_size=900, overlap=150):
    """Simple character-based chunking with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = []
for d in documents:
    for j, chunk in enumerate(chunk_text(d["text"])):
        chunks.append({
            "id": f"{d['doc_id']}_p{d['page']}_c{j}",
            "text": chunk,
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "chunk_id": j,
        })

print(f"Created {len(chunks)} chunks")
print(chunks[0]["id"])
print(textwrap.shorten(chunks[0]["text"], width=600))

## Extract knowledge graph triples

A knowledge graph triple has the form:

```text
(source entity) -[relationship]-> (target entity)
```

Examples:

```text
GraphRAG -[uses]-> knowledge graph
RAG -[combines]-> parametric memory
RAG -[combines]-> non-parametric memory
```

We use an LLM as an extraction engine. The system prompt tells it to return only JSON so Python can parse it.

In [ ]:
# ============================================================
# 6. Entity and relationship extraction
# ============================================================
# We only extract from the first N chunks to keep the demo affordable and fast.
# Increase MAX_EXTRACTION_CHUNKS for a larger graph.

import json, re
from tqdm.auto import tqdm

EXTRACT_SYS = "You extract knowledge graphs from technical documents. Return ONLY valid JSON."
EXTRACT_PROMPT = """
Extract important entities and relationships from the text.
Return JSON exactly in this shape:
{
  "triples": [
    {"source": "...", "relation": "...", "target": "...", "evidence": "short quote or phrase"}
  ]
}

Rules:
- Use canonical entity names.
- Keep relation names short, lowercase, and snake_case.
- Prefer technical concepts, methods, datasets, systems, tasks, papers, and authors.
- Do not invent facts not supported by the text.
- If no useful facts exist, return {"triples": []}.

Text:
{text}
"""


def clean_json_text(raw):
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()
    return raw


def extract_triples_from_chunk(chunk):
    raw = llm(EXTRACT_PROMPT.format(text=chunk["text"]), system=EXTRACT_SYS, temperature=0)
    raw = clean_json_text(raw)
    try:
        data = json.loads(raw)
        triples = data.get("triples", [])
    except Exception:
        triples = []

    cleaned = []
    for tr in triples:
        s = str(tr.get("source", "")).strip()
        r = str(tr.get("relation", "")).strip().lower().replace(" ", "_")
        t = str(tr.get("target", "")).strip()
        if s and r and t and s.lower() != t.lower():
            cleaned.append({
                "source": s,
                "relation": r,
                "target": t,
                "evidence": str(tr.get("evidence", "")).strip()[:500],
                "doc_id": chunk["doc_id"],
                "title": chunk["title"],
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"],
                "chunk_text": chunk["text"][:1500],
            })
    return cleaned

MAX_EXTRACTION_CHUNKS = 25
all_triples = []
for chunk in tqdm(chunks[:MAX_EXTRACTION_CHUNKS]):
    all_triples.extend(extract_triples_from_chunk(chunk))

print("Extracted triples:", len(all_triples))
for tr in all_triples[:20]:
    print(f"({tr['source']}) -[{tr['relation']}]-> ({tr['target']}) | {tr['doc_id']} p.{tr['page']}")

## Preview the graph locally with NetworkX

Before loading into Neo4j, it is useful to inspect the graph locally.

NetworkX keeps the graph in Python memory. Neo4j stores the graph in a database and supports Cypher queries, indexing, persistence, and graph operations at larger scale.

In [ ]:
# ============================================================
# 7. Build a local NetworkX preview graph
# ============================================================

import networkx as nx

G = nx.DiGraph()
for tr in all_triples:
    G.add_node(tr["source"], label="Entity")
    G.add_node(tr["target"], label="Entity")
    G.add_edge(
        tr["source"],
        tr["target"],
        relation=tr["relation"],
        doc_id=tr["doc_id"],
        page=tr["page"],
    )

print("Entities:", G.number_of_nodes())
print("Relationships:", G.number_of_edges())
print("Sample entities:", sorted(list(G.nodes()))[:30])

## Configure Neo4j

You can use either:

1. **Neo4j Aura** cloud database, or
2. local Neo4j running elsewhere.

Set these in Colab Secrets:

- `NEO4J_URI`, example: `neo4j+s://xxxxx.databases.neo4j.io`
- `NEO4J_USERNAME`, usually `neo4j`
- `NEO4J_PASSWORD`

The notebook will create `:Entity` nodes and `:RELATES_TO` relationships. The actual relation type is stored as a property called `relation` so we can support dynamic relation names safely.

In [ ]:
# ============================================================
# 8. Connect to Neo4j
# ============================================================

from neo4j import GraphDatabase

try:
    from google.colab import userdata
    NEO4J_URI = userdata.get("NEO4J_URI") or os.environ.get("NEO4J_URI", "")
    NEO4J_USERNAME = userdata.get("NEO4J_USERNAME") or os.environ.get("NEO4J_USERNAME", "neo4j")
    NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD") or os.environ.get("NEO4J_PASSWORD", "")
except Exception:
    NEO4J_URI = os.environ.get("NEO4J_URI", "")
    NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
    NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("Neo4j credentials not found. Add Colab Secrets to run the Neo4j cells.")
    driver = None
else:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session() as session:
        print(session.run("RETURN 'Neo4j connected' AS msg").single()["msg"])

In [ ]:
# ============================================================
# 9. Load triples into Neo4j
# ============================================================
# This creates:
#   (:Entity {name: source})-[:RELATES_TO {relation, evidence, source...}]->(:Entity {name: target})

if driver is None:
    print("Skipping Neo4j load because driver is not configured.")
else:
    with driver.session() as session:
        session.run("CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")

        # Optional: clean previous demo data.
        session.run("MATCH (n:Entity) DETACH DELETE n")

        query = """
        UNWIND $rows AS row
        MERGE (s:Entity {name: row.source})
        MERGE (t:Entity {name: row.target})
        MERGE (s)-[r:RELATES_TO {relation: row.relation, doc_id: row.doc_id, page: row.page, chunk_id: row.chunk_id}]->(t)
        SET r.evidence = row.evidence,
            r.title = row.title,
            r.chunk_text = row.chunk_text
        """
        session.run(query, rows=all_triples)

        counts = session.run("""
        MATCH (n:Entity) WITH count(n) AS nodes
        MATCH ()-[r:RELATES_TO]->() RETURN nodes, count(r) AS rels
        """).single()
        print("Neo4j graph loaded:", dict(counts))

## Query the graph with Cypher

Cypher is Neo4j's graph query language.

The query below asks for the most connected entities. This helps you understand what concepts dominate the corpus.

In [ ]:
# ============================================================
# 10. Explore the graph
# ============================================================

if driver is None:
    print("Neo4j not configured.")
else:
    with driver.session() as session:
        rows = session.run("""
        MATCH (e:Entity)
        OPTIONAL MATCH (e)-[r]-()
        RETURN e.name AS entity, count(r) AS degree
        ORDER BY degree DESC
        LIMIT 15
        """).data()

    for row in rows:
        print(f"{row['entity']:55s} degree={row['degree']}")

## Graph retrieval: find paths and neighborhood facts

For GraphRAG, retrieval means **finding connected facts**, not nearest vectors.

The next function:

1. Finds graph entities mentioned in the question.
2. Retrieves nearby relationships around those entities.
3. Optionally finds paths between two entities.
4. Sends those facts to the LLM.

In [ ]:
# ============================================================
# 11. GraphRAG retrieval functions
# ============================================================

import re

def find_entity_candidates(question, max_results=10):
    """Find Neo4j entities whose names overlap with words in the question."""
    if driver is None:
        return []
    tokens = [t for t in re.findall(r"[A-Za-z][A-Za-z0-9_-]+", question) if len(t) > 2]
    if not tokens:
        return []
    pattern = "|".join(re.escape(t) for t in tokens)
    with driver.session() as session:
        rows = session.run("""
        MATCH (e:Entity)
        WHERE toLower(e.name) =~ toLower($pattern)
           OR any(tok IN $tokens WHERE toLower(e.name) CONTAINS toLower(tok))
        RETURN e.name AS name
        LIMIT $limit
        """, pattern=f".*({pattern}).*", tokens=tokens, limit=max_results).data()
    return [r["name"] for r in rows]


def graph_neighborhood(entity_names, hops=2, limit=80):
    """Return relationship facts within N hops of selected entities."""
    if driver is None or not entity_names:
        return []
    with driver.session() as session:
        rows = session.run("""
        MATCH (seed:Entity)
        WHERE seed.name IN $names
        MATCH path=(seed)-[*1..2]-(neighbor)
        WITH relationships(path) AS rels
        UNWIND rels AS r
        WITH DISTINCT startNode(r).name AS source,
             r.relation AS relation,
             endNode(r).name AS target,
             r.doc_id AS doc_id,
             r.page AS page,
             r.evidence AS evidence
        RETURN source, relation, target, doc_id, page, evidence
        LIMIT $limit
        """, names=entity_names, limit=limit).data()
    return rows


def graph_answer(question, hops=2):
    entities = find_entity_candidates(question)
    facts = graph_neighborhood(entities, hops=hops)
    fact_text = "\n".join(
        f"{f['source']} -[{f['relation']}]-> {f['target']} (source: {f['doc_id']} p.{f['page']}; evidence: {f.get('evidence','')})"
        for f in facts
    ) or "No graph facts found."

    prompt = f"""
Use the knowledge graph facts below to answer the question.
Trace relationships step by step when useful.
If the graph does not contain enough evidence, say what is missing.

Seed entities: {entities}

Facts:
{fact_text}

Question: {question}
Answer:
"""
    answer = llm(prompt)
    print("Seed entities:", entities)
    print("Facts retrieved:", len(facts))
    print("\nAnswer:\n", answer)
    return answer, facts

_ = graph_answer("How are RAG and GraphRAG related?", hops=2)

## Teaching recap

GraphRAG workflow:

```text
PDFs → chunks → LLM triple extraction → Neo4j graph → graph retrieval → LLM answer
```

Main advantage:

- Strong for relationship, path, and corpus-structure questions.

Main limitation:

- Requires extraction quality, entity resolution, graph schema design, and database setup.